# SDS — NER

Run NER on OCR output, evaluate, and save results.

**Requires:** OCR cache populated by `ocr.ipynb`

GPU models also need: `pip install torch transformers vllm`

## 1. Configuration

In [ ]:
import sys
from pathlib import Path

PROJECT = Path(".").resolve()
if not (PROJECT / "synthdoc").exists():
    PROJECT = PROJECT.parent
DATA_DIR = PROJECT / "data"
RESULTS_DIR = DATA_DIR / "results"

sys.path.insert(0, str(PROJECT))

In [ ]:
OCR_ENGINE = "glm"      # glm | tesseract | qwenvl | deepseek
NER_ENGINES = ["qwen3"] # spacy | deberta | qwen3

## 2. Load Pages + OCR Cache

In [ ]:
from synthdoc.bench.helpers import load_ner_pages, FULL_ENTITY_TYPES
from synthdoc.bench.config import load_engines, _resolve
from synthdoc.bench.run_benchmark import _load_ocr_cache

pages = load_ner_pages(DATA_DIR, entity_types=FULL_ENTITY_TYPES)
ocr_key = f"{OCR_ENGINE}_ocr"
_load_ocr_cache(pages, ocr_key, DATA_DIR)

cached = sum(1 for p in pages if ocr_key in p)
print(f"Loaded {len(pages)} pages ({cached} with {OCR_ENGINE} OCR cache)")
if cached < len(pages):
    print(f"  WARNING: {len(pages) - cached} pages missing OCR — run ocr.ipynb first")

engines_config = load_engines()

## 3. Run NER

In [ ]:
import json
from synthdoc.bench.config import _dispatch_ner
from synthdoc.bench.helpers import InferenceTimer

NER_DIR = DATA_DIR / "ner"
NER_DIR.mkdir(exist_ok=True)

all_ner_results = {}

for ner_name in NER_ENGINES:
    model_label = f"{OCR_ENGINE}_{ner_name}"
    runner, model_id = _resolve(ner_name, "ner", engines_config)

    remaining = [p for p in pages if not (NER_DIR / f"{p['stem']}_{ner_name}.json").exists()]
    print(f"Running {model_label}: {len(remaining)} pages ({len(pages) - len(remaining)} already done)...")

    with InferenceTimer(f"{model_label}_ner") as timer:
        ner_key = _dispatch_ner(runner, remaining, ocr_key, model_id=model_id, ner_dir=NER_DIR, ner_engine=ner_name)

    all_ner_results[ner_name] = ner_key
    print(f"  Done in {timer.result['time_s']}s  (ner_key={ner_key})")

## 4. Load NER Results

In [ ]:
# Load per-page NER results back onto page dicts for evaluate step
import json

for ner_name in NER_ENGINES:
    ner_key = all_ner_results[ner_name]
    for p in pages:
        if ner_key not in p:
            path = NER_DIR / f"{p['stem']}_{ner_name}.json"
            if path.exists():
                p[ner_key] = json.loads(path.read_text())["entities"]
            else:
                p[ner_key] = []

In [ ]:
import pandas as pd
from synthdoc.bench.run_benchmark import evaluate_model

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for ner_name in NER_ENGINES:
    ner_key = all_ner_results[ner_name]
    model_label = f"{OCR_ENGINE}_{ner_name}"
    results = evaluate_model(pages, model_label, ner_key, ocr_key)
    all_results.append(results)

    for eset in ["standard", "full"]:
        subset = results["ner_df"][results["ner_df"]["entity_set"] == eset]
        n = 7 if eset == "standard" else 14
        print(f"\n{model_label} [{eset} - {n} types]:")
        print(f"  Strict F1:  {subset['strict_f1'].mean():.4f}")
        print(f"  Relaxed F1: {subset['relaxed_f1'].mean():.4f}")

In [ ]:
combined = pd.concat([r["ner_df"] for r in all_results], ignore_index=True)

print("Summary (standard):")
std = combined[combined["entity_set"] == "standard"]
summary = std.groupby("ocr").agg({
    "strict_f1": "mean",
    "relaxed_f1": "mean",
    "strict_precision": "mean",
    "strict_recall": "mean",
}).round(4)
print(summary.to_string())

## 5. Save Results

In [ ]:
# Save per-entity matches (used by build_assets.py)
entity_dfs = [r["entity_matches"] for r in all_results if not r["entity_matches"].empty]
if entity_dfs:
    import pandas as pd
    entity_combined = pd.concat(entity_dfs, ignore_index=True)
    entity_combined.to_csv(DATA_DIR / "entity_matches.csv", index=False)
    print(f"Saved entity_matches.csv ({len(entity_combined)} rows) to {DATA_DIR}/")

# Save aggregated benchmark results
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
combined.to_csv(RESULTS_DIR / "benchmark_results.csv", index=False)

for r in all_results:
    model = r["model"]
    r["ner_df"].to_csv(RESULTS_DIR / f"benchmark_{model}.csv", index=False)
    r["type_df"].to_csv(RESULTS_DIR / f"per_type_{model}.csv", index=False)

print(f"Saved detailed results to {RESULTS_DIR}/")

## 6. Spot-Check Failures

In [ ]:
from synthdoc.bench.helpers import detailed_match, NER_ENTITY_TYPES

ner_name = NER_ENGINES[0]
ner_key = all_ner_results[ner_name]
model_label = f"{OCR_ENGINE}_{ner_name}"

print(f"Missed entities for {model_label}:\n")

misses = []
for p in pages:
    gt = [e for e in p["gt_entities"] if e["type"] in NER_ENTITY_TYPES]
    pred = p.get(ner_key, [])
    _, unmatched_gt, _ = detailed_match(gt, pred)
    for etype, text in unmatched_gt:
        misses.append({"template": p["template_type"], "type": etype, "text": text})

if misses:
    miss_df = pd.DataFrame(misses)
    print(f"Total missed: {len(miss_df)}")
    print(f"\nBy type:")
    print(miss_df["type"].value_counts().to_string())
    print(f"\nBy template:")
    print(miss_df["template"].value_counts().to_string())
    print(f"\nExamples:")
    print(miss_df.head(20).to_string(index=False))
else:
    print("No misses - perfect recall!")